[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/FutoshiNakamura-Tts/gaussian-splatting-colab/blob/colab-t4-2025-12-18/gaussian_splatting_colab.ipynb)

## 1. Environment Check

In [ ]:
!nvidia-smi
!python --version
!nvcc --version

## 2. Install Dependencies

In [ ]:
%cd /content
!git clone --recursive https://github.com/camenduru/gaussian-splatting
!pip install -q plyfile

In [ ]:
%cd /content/gaussian-splatting
import sys
import subprocess

def install_package(package_name, wheel_glob_pattern, source_path):
    # 1. Search for local wheel in project 'wheels/' directory
    import glob
    local_wheels = glob.glob(f"/content/gaussian-splatting/wheels/{wheel_glob_pattern}")
    
    if local_wheels:
        wheel_path = local_wheels[0]
        print(f"Found local wheel for {package_name}: {wheel_path}")
        try:
            subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", wheel_path])
            print(f"Successfully installed {package_name} from local wheel.")
            return
        except subprocess.CalledProcessError:
            print(f"Failed to install local wheel for {package_name}. Moving to fallback...")
    else:
        print(f"No local wheel found for {package_name} in wheels/ directory.")
    
    # 2. Fallback: Build from source
    print(f"Building {package_name} from source: {source_path}")
    try:
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", source_path])
        print(f"Successfully installed {package_name} from source.")
    except subprocess.CalledProcessError as e:
        print(f"Failed to install {package_name} from source.")
        raise e

# Packages configuration
packages = [
    {
        "name": "diff-gaussian-rasterization",
        "wheel_glob_pattern": "diff_gaussian_rasterization-*-cp310-*-linux_x86_64.whl",
        "source_path": "/content/gaussian-splatting/submodules/diff-gaussian-rasterization"
    },
    {
        "name": "simple-knn",
        "wheel_glob_pattern": "simple_knn-*-cp310-*-linux_x86_64.whl",
        "source_path": "/content/gaussian-splatting/submodules/simple-knn"
    }
]

for pkg in packages:
    install_package(pkg["name"], pkg["wheel_glob_pattern"], pkg["source_path"])

## 3. Download Data

In [ ]:
!wget https://huggingface.co/camenduru/gaussian-splatting/resolve/main/tandt_db.zip
!unzip tandt_db.zip

## 4. Training

In [ ]:
!python train.py -s /content/gaussian-splatting/tandt/train